# 第 0 章：从一次错误回答认识工程系统

**适合读者**：会写 Python，了解大语言模型（LLM, Large Language Model）的提示词和工具调用，准备把智能体（Agent）接入业务知识与真实数据。

**本章目标**：解释检索增强生成（RAG, Retrieval-Augmented Generation）的价值；画出索引期与查询期；分清知识定义、业务事实和模型生成；能从本项目找到每一层代码。

假设用户问：“2026 年 8 月华东地区的净销售额是多少？”通用模型也许知道“净销售额”这个词，却不知道本公司的退款口径，更不知道数据库里当月的支付记录。直接让模型猜一个金额，回答可能流畅却无法核对。

![RAG 工程链：知识准备、检索生成与评估反馈](assets/rag_pipeline.svg)


## 0.1 为什么需要检索

本项目把问题拆成三类证据：

| 要回答的部分 | 证据在哪里 | 例子 |
|---|---|---|
| “净”如何计算 | 经审核的知识卡 | 只扣已批准的退款，时间窗口按支付时间 |
| 华东关联哪些表 | 表说明和连接规则 | 订单通过客户关联地区 |
| 实际金额是多少 | PostgreSQL 业务表 | 成功支付与退款的真实记录 |

结构化查询语言（SQL, Structured Query Language）负责在业务表上计算。自然语言到结构化查询语言（NL2SQL, Natural Language to SQL）系统负责把用户意图变成可核验的查询。RAG 为这一步补充业务定义和表结构；它不能把知识卡当成金额数据库。

> **一句话记忆**：知识库解释“怎么算”，数据库提供“算什么”，模型组织步骤和文字。

## 0.2 两条生命周期

~~~mermaid
flowchart LR
  subgraph Index["索引期：知识发生变化时"]
    S["经审核的业务规则与表说明"] --> C["语义单元、来源、版本"]
    C --> E["文档向量"]
    E --> V[("rag.documents")]
  end
  subgraph Ask["查询期：每次提问"]
    Q["用户问题"] --> A["智能体"]
    A --> R["检索相关知识"]
    R --> V
    V --> A
    A --> X["生成并校验 SQL"]
    X --> D[("analytics 业务表")]
    D --> O["带证据的回答"]
  end
~~~

索引期处理资料、生成嵌入并发布索引；查询期理解问题、召回候选、组织上下文、生成查询并核对结果。索引若过期，检索会找到旧定义；查询若没有执行，模型就没有当月金额。两条链需要分别测试。

## 0.3 为什么工程上有多种路线

| 路线 | 适用情况 | 优点 | 主要限制 |
|---|---|---|---|
| 把全部资料放进长提示词 | 很小、变化慢的知识集 | 无索引服务，原型快 | 上下文变长，成本与噪声增加 |
| 固定流程 RAG | 单一知识源的问答 | 路径稳定、易测试 | 多步任务需要额外编排 |
| 智能体式 RAG | 需要查知识、验表结构、执行工具 | 能按问题选择工具 | 调用路径多，更依赖轨迹评估 |
| 微调模型 | 稳定的表达格式或行为 | 可改善固定任务表现 | 不代替实时业务事实与权限控制 |
| 直接查询数据库 | 指标定义已明确 | 数值准确、可重算 | 自然语言中的口径歧义仍需解决 |

项目使用谷歌智能体开发套件（ADK, Agent Development Kit）管理工具调用；Qwen 生成检索向量，DeepSeek 规划并生成 SQL，pgvector 与 PostgreSQL 保存知识索引和业务事实。选择这些组件并不意味着其他框架不可用：先用可靠的问题集测正确率、延迟和成本，再决定是否换方案。

## 0.4 先离线找到项目地图

下面的单元只读取文件，不连接数据库，也不调用外部应用程序编程接口（API, Application Programming Interface）。从项目根目录或 <code>tutorial</code> 文件夹打开 Notebook 都能运行。首次运行可在项目根目录执行 <code>uv sync --extra dev</code> 与 <code>uv run jupyter lab</code>；请在 Jupyter 中选择本项目虚拟环境的 Python 内核。

In [1]:
from pathlib import Path
import json

root = Path.cwd()
if root.name == "tutorial":
    root = root.parent
root = root.resolve()
assert (root / "db" / "knowledge.json").is_file(), f"未找到项目根目录：{root}"
print("项目根目录：", root)

files = {
    "业务知识": "db/knowledge.json",
    "索引": "src/nl2sql/indexer.py",
    "检索": "src/nl2sql/retrieval.py",
    "嵌入": "src/nl2sql/embedding.py",
    "安全执行": "src/nl2sql/sql_runner.py",
    "智能体": "nl2sql_agent/agent.py",
}
for role, relative in files.items():
    print(f"{role:8} -> {relative}: {(root / relative).is_file()}")

项目根目录： D:\Projects\PythonProject\nl2sql-rag
业务知识     -> db/knowledge.json: True
索引       -> src/nl2sql/indexer.py: True
检索       -> src/nl2sql/retrieval.py: True
嵌入       -> src/nl2sql/embedding.py: True
安全执行     -> src/nl2sql/sql_runner.py: True
智能体      -> nl2sql_agent/agent.py: True


知识卡使用 JavaScript 对象表示法（JSON, JavaScript Object Notation）保存。每张卡有稳定标识符（ID, Identifier）、类型、别名、正文和依赖表。下面数一数卡片，并只打印短摘要；真实密钥、连接字符串和个人数据都不应输出到 Notebook。

In [2]:
from collections import Counter

cards = json.loads((root / "db" / "knowledge.json").read_text(encoding="utf-8"))
print("知识卡总数：", len(cards))
print("类型分布：", dict(Counter(card["kind"] for card in cards)))
for card in cards[:3]:
    print(card["id"], "->", card["content"][:45] + "…")
assert any(card["id"] == "metric.net_sales" for card in cards)

知识卡总数： 13
类型分布： {'table': 6, 'metric': 3, 'join': 2, 'rule': 1, 'example': 1}
table.customers -> 客户表 analytics.customers：customer_id 是主键；regio…
table.products -> 商品表 analytics.products：product_id 是主键；categor…
table.orders -> 订单表 analytics.orders：order_id 是主键；customer_id…


## 0.5 用错误现象定位层次

| 现象 | 优先检查 |
|---|---|
| “净销售额”定义错误 | 知识卡是否正确、是否索引了新版本、必需文档是否召回 |
| 表或字段凭空出现 | 检索上下文与实时表结构核验 |
| 查询返回金额错误 | 时间字段、连接基数、退款状态、数据库执行结果 |
| 回答正确但没有执行查询 | 智能体工具轨迹；这可能只是偶然猜中 |
| 向量服务不可用 | 明确报告检索失败，不凭记忆给出数字 |

第 1 章准备知识，第 2 章建立向量索引的心智模型，第 3 章比较召回与融合，后续章节再把证据交给智能体、约束 SQL、评估和部署。

## 练习

**基础练习**：以“2026 年 8 月华东净销售额”为例，写出三行追踪记录：① 哪张知识卡定义口径；② 哪张卡或表说明给出地区连接；③ 哪一步必须访问业务数据库。**检查标准**：至少指出 <code>metric.net_sales</code>、<code>join.customer_orders</code> 和实际 SQL 执行；不得把知识卡中的示例金额当成查询结果。

**进阶练习**：假设最终回答把“已批准退款”误写成“所有退款”。提出一个可重复的排查顺序，并分别给索引期、检索期、生成/执行期安排一条验证。**检查标准**：每条验证都应有可观察的输入和输出，例如文档 ID、工具调用、执行 SQL 或独立计算的数值；不能只写“优化提示词”。

## 继续阅读

- [Microsoft 的 RAG 方案设计与评估](https://learn.microsoft.com/en-us/azure/architecture/ai-ml/guide/rag/rag-solution-design-and-evaluation-guide)
- [pgvector 官方文档](https://github.com/pgvector/pgvector)

本章无需模型密钥。真正运行项目服务时，再按根目录 README 完成数据库迁移、数据初始化和索引构建。